In [1]:
import os
os.chdir("..")

from data.data import ssl_ds
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

# --- transforms (no normalization!) ---
to_tensor_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),          # [0,1]
])

def transform_batch(batch):
    # batch["image"] is a list of PIL Images when DataLoader passes a list of indices
    imgs = batch["image"]
    batch["image"] = [to_tensor_224(img) for img in imgs]
    return batch

@torch.no_grad()
def get_mean_std(loader):
    channels_sum = torch.zeros(3)
    channels_sq_sum = torch.zeros(3)
    num_pixels = 0

    for batch in loader:
        data = batch["image"]              # usually a Tensor [B,C,H,W] after default collate
        if isinstance(data, list):         # safeguard in case it stays a list
            data = torch.stack(data, dim=0)

        b, c, h, w = data.shape
        channels_sum     += data.sum(dim=[0, 2, 3])
        channels_sq_sum  += (data ** 2).sum(dim=[0, 2, 3])
        num_pixels       += b * h * w

        assert c == 3 and h == 224 and w == 224, f"Expected input shape [B,3,224,224], got: {data.shape}"

    mean = channels_sum / num_pixels
    var  = (channels_sq_sum / num_pixels) - mean ** 2
    std  = torch.sqrt(var.clamp_min(0))
    return mean, std

@torch.no_grad()
def get_mean_std_masked(loader, thr=5/255):  # ignore near-black pixels
    ch_sum = torch.zeros(3)
    ch_sq  = torch.zeros(3)
    count  = torch.zeros(3)
    for batch in loader:
        x = batch["image"]
        if isinstance(x, list): x = torch.stack(x, 0)   # [B,3,224,224]
        # mask per-channel: keep pixels > thr
        mask = x > thr                                   # [B,3,H,W]
        ch_sum += (x * mask).sum(dim=[0,2,3])
        ch_sq  += ((x**2) * mask).sum(dim=[0,2,3])
        count  += mask.sum(dim=[0,2,3])
    mean = ch_sum / count.clamp_min(1)
    var  = (ch_sq / count.clamp_min(1)) - mean**2
    std  = var.clamp_min(0).sqrt()
    kept = (count / (len(loader.dataset)*224*224)).cpu().numpy()
    return mean, std, kept  # kept = fraction of non-near-black pixels per channel


ds_train = ssl_ds["train"].with_transform(transform_batch)

loader = DataLoader(
    ds_train,
    batch_size=128,
    shuffle=False,
    num_workers=32,
    pin_memory=True,   # optional
    # no custom collate_fn; default stacks tensors for us
)

mean, std = get_mean_std(loader)
print(f"Calculated Mean: {mean}")
print(f"Calculated Std:  {std}")

mean_m, std_m, kept = get_mean_std_masked(loader)
print(f"Masked Mean: {mean_m}")
print(f"Masked Std:  {std_m}")
print(f"Fraction of non-near-black pixels per channel: {kept}")


Calculated Mean: tensor([0.1678, 0.1629, 0.1592])
Calculated Std:  tensor([0.1225, 0.1134, 0.1062])
Masked Mean: tensor([0.1687, 0.1631, 0.1596])
Masked Std:  tensor([0.1223, 0.1133, 0.1061])
Fraction of non-near-black pixels per channel: [0.99390227 0.99831617 0.9972756 ]


In [ ]:
"""
Foreground-only mean/std for astronomy images using Otsu (with fallbacks).
- Threshold on luma (0.299R + 0.587G + 0.114B).
- Ignores exact zeros in histogram to avoid black-dominated Otsu.
- Light morphology to remove tiny specks; holes filled to keep galaxy cores intact.
- Computes micro-average (over all fg pixels in the dataset).

Requires: numpy, torch, torchvision, scikit-image (skimage), optionally scipy (for binary_fill_holes fallback).
"""

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

# --- optional deps (skimage strongly recommended) ---
try:
    from skimage.filters import threshold_otsu, threshold_triangle, threshold_yen
    from skimage.morphology import remove_small_objects, remove_small_holes, binary_opening, disk
    SKIMAGE_OK = True
except Exception:
    SKIMAGE_OK = False
    from scipy.ndimage import binary_opening, binary_fill_holes
    def threshold_triangle(x):  # crude fallback
        return float(np.quantile(x, 0.10))
    def threshold_yen(x):
        return float(np.quantile(x, 0.20))
    def threshold_otsu(x):
        # very rough fallback if skimage is missing
        hist, bins = np.histogram(x, bins=256, range=(0, 1))
        cdf = np.cumsum(hist)
        total = cdf[-1]
        sumB, wB, maximum, sum1 = 0.0, 0.0, 0.0, np.dot(hist, (bins[:-1] + bins[1:]) / 2)
        for i in range(256):
            wB += hist[i]
            if wB == 0:
                continue
            wF = total - wB
            if wF == 0:
                break
            sumB += (bins[i] + bins[i+1]) / 2 * hist[i]
            mB = sumB / wB
            mF = (sum1 - sumB) / wF
            between = wB * wF * (mB - mF) ** 2
            if between >= maximum:
                maximum = between
                level = (bins[i] + bins[i+1]) / 2
        return float(level)

to_tensor = transforms.ToTensor()

def rgb_to_luma(t):  # t: CxHxW tensor [0,1]
    return 0.299 * t[0] + 0.587 * t[1] + 0.114 * t[2]

def make_fg_mask_luma(img_tensor,
                      min_fg_frac=0.005,
                      open_radius=1,
                      min_object_size=32,
                      hole_area=64):
    """
    Build a foreground mask from a [0,1] tensor image (CxHxW).
    Steps:
      1) compute luma
      2) compute threshold on nonzero luma using Otsu (fallbacks: Yen, Triangle)
      3) mask = luma > thresh
      4) light opening + remove tiny objects + fill small holes
    Returns: bool mask (HxW)
    """
    C, H, W = img_tensor.shape
    luma = rgb_to_luma(img_tensor).cpu().numpy()
    flat = luma.reshape(-1)

    # Exclude exact zeros from thresholding histogram (pure background)
    nz = flat[flat > 0]
    if nz.size < H * W * min_fg_frac:
        # image is near-empty or super black; relax by using a low percentile as threshold
        t = float(np.quantile(flat, 0.15))
    else:
        # Try Otsu; if it collapses to ~0, use Yen or Triangle
        t = threshold_otsu(nz)
        if t <= 1e-6 and nz.size > 0:
            # very dark images can break Otsu
            try:
                t = threshold_yen(nz)
            except Exception:
                t = threshold_triangle(nz)
        # guardrails
        t = float(np.clip(t, 0.0, 0.5))

    mask = (luma > t)

    # Morphology (very light to keep stars): opening with tiny disk, drop tiny crumbs, fill small holes
    if SKIMAGE_OK:
        if open_radius > 0:
            mask = binary_opening(mask, footprint=disk(open_radius))
        if min_object_size > 0:
            mask = remove_small_objects(mask, min_object_size)
        if hole_area > 0:
            mask = remove_small_holes(mask, area_threshold=hole_area)
    else:
        if open_radius > 0:
            mask = binary_opening(mask, iterations=open_radius)
        # crude small-object removal
        # (skip if no skimage; leaving mask as-is is safer than over-pruning)
        if hole_area > 0:
            mask = binary_fill_holes(mask)

    # If we somehow lost almost everything, fall back to a lenient mask
    if mask.mean() < min_fg_frac:
        p = np.quantile(nz if nz.size else flat, 0.70)
        mask = (luma > p)

    return mask

def compute_foreground_mean_std(dataset,
                                open_radius=1,
                                min_object_size=32,
                                hole_area=64,
                                min_fg_frac=0.005):
    """
    Compute channel-wise mean/std over foreground pixels only.
    Returns: mean(list of 3 floats), std(list of 3 floats), coverage (avg fg fraction per image)
    """
    sum_c = np.zeros(3, dtype=np.float64)
    sumsq_c = np.zeros(3, dtype=np.float64)
    total_fg = 0
    fg_fracs = []

    for batch in dataset:
        imgs = batch[0] if isinstance(batch, (tuple, list)) else batch["image"]

        # support list[PIL], PIL, or tensor
        if isinstance(imgs, list):
            tensors = [to_tensor(im) for im in imgs]
            imgs_t = torch.stack(tensors, dim=0)
        elif torch.is_tensor(imgs):
            imgs_t = imgs.float()
            if imgs_t.max() > 1.0:  # likely uint8 0–255
                imgs_t = imgs_t / 255.0
        else:
            # single PIL
            imgs_t = to_tensor(imgs).unsqueeze(0)

        B, C, H, W = imgs_t.shape
        imgs_t = imgs_t.clamp(0, 1)

        for i in range(B):
            x = imgs_t[i]  # CxHxW
            mask = make_fg_mask_luma(
                x, min_fg_frac=min_fg_frac,
                open_radius=open_radius,
                min_object_size=min_object_size,
                hole_area=hole_area
            )
            fg_fracs.append(float(mask.mean()))
            if mask.any():
                fg = x[:, mask]  # 3 x N_fg
                arr = fg.cpu().numpy().astype(np.float64)
                sum_c += arr.mean(axis=1) * arr.shape[1]  # accumulate as micro-average
                sumsq_c += ((arr ** 2).mean(axis=1) * arr.shape[1])
                total_fg += arr.shape[1]

    if total_fg == 0:
        raise RuntimeError("No foreground pixels detected. Check thresholds/morphology settings.")

    mean = (sum_c / total_fg).astype(np.float64)
    var = (sumsq_c / total_fg) - mean**2
    var[var < 1e-12] = 1e-12
    std = np.sqrt(var)

    return mean.tolist(), std.tolist(), float(np.mean(fg_fracs))

# Example usage:
mean, std, cov = compute_foreground_mean_std(loader)
print(f"Calculated Mean (Otsu): {mean}")
print(f"Calculated Std (Otsu):  {std}")
print(f"Average foreground pixel fraction: {cov}")


Calculated Mean (Otsu): [0.5563723577581228, 0.5466399687071238, 0.5065107300061998]
Calculated Std (Otsu):  [0.20517936480092377, 0.1763141906401364, 0.16977451864162768]
Average foreground pixel fraction: 0.0543530364146896
